# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

- **Citation:** Kulka, M., Rodriguez Miera, S. and Marcet-Palacios, M. 2026. Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells. Frontiers.
- **Identifier:** 10.71728/senscience.vq4a-28xa


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}\n")
print(f"Published: {metadata.date_published}")
print(f"Authors: {metadata.author}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll list all record sets, their `@id`, and their available fields and columns by `@id`. This allows precise referencing for data extraction and processing. This dataset may have multiple record sets.

In [ ]:
# List all record sets and their fields by '@id'
record_sets = dataset.metadata.record_sets
print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"RecordSet Name: {rs.name} | @id: {rs.id}")
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields:")
        for field in rs.fields:
            print(f"    Field name: {field.name} | @id: {field.id} | Data type: {getattr(field, 'data_type', 'n/a')}")
    if hasattr(rs, 'columns') and rs.columns:
        print("  Columns:")
        for col in rs.columns:
            print(f"    Column name: {col.name} | @id: {col.id} | Data type: {getattr(col, 'data_type', 'n/a')}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field/column `@id`s from the overview.

Below, we'll extract data for **all record sets** into DataFrames for further processing. Update `'@id'` values as needed.

In [ ]:
# Extract data from each record set
dataframes = {}
all_record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
print(f"Loading record sets (by @id): {all_record_set_ids}")

for record_set_id in all_record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from {record_set_id}")
        if not df.empty:
            print(f"Sample columns: {df.columns.tolist()}")
            display(df.head())
    except Exception as e:
        print(f"Could not load data for record set {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Below, we'll select a numeric field (by `@id`) from one of the loaded record sets, filter, normalize, and group it.

_Adjust the selected record set and field as needed for your data!_

In [ ]:
# Example: Select a record set and fields by '@id'.
# For demonstration, let's use the first non-empty record set and a numeric field.

# Find a suitable record set and a numeric field
selected_record_set_id = None
numeric_field_id = None
group_field_id = None

for rs in dataset.metadata.record_sets:
    this_id = rs.id
    df = dataframes.get(this_id, pd.DataFrame())
    if not df.empty:
        # Find a numeric field
        numeric_candidates = []
        if hasattr(rs, 'fields'):
            for f in rs.fields:
                if hasattr(f, 'data_type') and f.data_type in ['Float', 'Integer', 'Number'] and f.id in df.columns:
                    numeric_candidates.append(f.id)
        if hasattr(rs, 'columns'):
            for c in rs.columns:
                if hasattr(c, 'data_type') and c.data_type in ['Float', 'Integer', 'Number'] and c.id in df.columns:
                    numeric_candidates.append(c.id)
        if numeric_candidates:
            selected_record_set_id = this_id
            numeric_field_id = numeric_candidates[0]
            # Try to find a grouping field (prefer string/categorical, but fallback to any field)
            for f in (getattr(rs, 'fields', []) + getattr(rs, 'columns', [])):
                if getattr(f, 'data_type', '') in ['Text', 'String'] and f.id in df.columns:
                    group_field_id = f.id
                    break
            if not group_field_id and df.columns.size > 1:
                group_field_id = df.columns[1]  # fallback option
            break

print(f"Selected record set: {selected_record_set_id}")
print(f"Selected numeric field: {numeric_field_id}")
print(f"Selected group field: {group_field_id}")

if selected_record_set_id and numeric_field_id and numeric_field_id in dataframes[selected_record_set_id].columns:
    df = dataframes[selected_record_set_id]
    # Ensure numeric conversion
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].quantile(0.5) # median as threshold
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (median): {filtered_df.shape[0]} records")
    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Grouping
    if group_field_id in df.columns:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        print(grouped.head())
else:
    print("No suitable record set and field found for EDA. Please check your data overview and select appropriate fields.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We use matplotlib/seaborn to plot the distribution of the selected numeric field and the group means, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id and numeric_field_id and numeric_field_id in dataframes[selected_record_set_id].columns:
    df = dataframes[selected_record_set_id]
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in {selected_record_set_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    
    # Grouped plot if group field exists
    if group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        order = df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False).index[:10]
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df[df[group_field_id].isin(order)], order=order)
        plt.title(f"Distribution of {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No visualization available. Select a record set and numeric field in previous cells.")

## 6. Conclusion
In this notebook, we've loaded and explored the FAIR² dataset, identified the available record sets, and performed basic data extraction and exploratory analysis using the `mlcroissant` library.

- **Key metadata fields, record sets, and their fields/columns are referenced by their `@id` for reliable access.**
- **We demonstrated filtering, normalization, grouping, and data visualization.**

Continue your analysis by selecting additional fields/features using their `@id`, and by customizing filtering or transformation steps as needed.